# 06. Feature Extraction

1. Data Loading
2. Feature Definitions
3. Feature Computation
   - 3.1 English Frequency
   - 3.2 Lexical Diversity (TTR, MTLD)
   - 3.3 English Phrase Length
   - 3.4 Code-Switching
   - 3.5 Components of Lexical Diversity
4. Feature Checks
   - 4.1 Summary Statistics
   - 4.2 Length Sensitivity of TTR and MTLD
   - 4.3 MTLD Implementation Check
   - 4.4 Preview by Generation and Period
5. Export

## 1. Data Loading

### Objective
Load the labeled songs (`04`), cleaned lyrics and tokens (`05`), and keep songs usable for lyric analysis.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

INTERIM_DIR = Path("../data/interim")

songs = pd.read_pickle(INTERIM_DIR / "songs_labeled.pkl")
lyrics = pd.read_pickle(INTERIM_DIR / "lyrics_clean.pkl")
tokens = pd.read_pickle(INTERIM_DIR / "tokens.pkl")

songs = songs.merge(lyrics[["song_key", "exclude_lyrics"]], on="song_key", how="left")
songs = songs[~songs["exclude_lyrics"].fillna(True)].reset_index(drop=True)

# Only Korean and English tokens are used; numbers and Japanese/Chinese tokens are dropped
tokens = tokens[tokens["lang"].isin(["ko", "en"])]
tokens = tokens.sort_values(["song_key", "line_no", "token_no"]).reset_index(drop=True)

print(f"Songs for analysis: {len(songs):,}")
print(f"Korean/English tokens: {len(tokens):,}")

Songs for analysis: 7,984
Korean/English tokens: 1,666,993


## 2. Feature Definitions

All features are computed per song in three versions:
- `_all`: all tokens (main version)
- `_nofill`: English fillers (`oh`, `yeah`, `la`, ...) removed before computing
- `_romko`: romanized Korean tokens (`is_romanized`, `05` 6.2) counted as Korean instead of English

| Feature | Definition | Songs |
|---|---|---|
| `en_ratio` | English tokens / (Korean + English tokens) | All |
| `ttr` | Distinct English words / English tokens | English tokens ≥ `MIN_EN_TOKENS` |
| `mtld` | Mean length of English word runs that keep TTR above 0.72 (forward and backward average) | English tokens ≥ `MIN_EN_TOKENS` |
| `phrase_len_mean` | Mean length of consecutive English token runs within a line | At least one English token |
| `phrase_len_max` | Longest English run within a line | At least one English token |
| `cs_per100` | Korean↔English switches **within lines** per 100 tokens | All |
| `cs_inter_per100` | Korean↔English switches **at line breaks** per 100 tokens | All |
| `cs_total_per100` | All Korean↔English switches (within lines + at line breaks) per 100 tokens | All |
| `mixed_line_ratio` | Lines containing both Korean and English / all lines | All |

- TTR is intuitive but decreases as texts get longer, so MTLD, which is robust to text length, is reported alongside it.
- `MIN_EN_TOKENS` applies only to TTR and MTLD; songs below it get NaN for these two features and stay in all other analyses.
- `ttr_raw` and `mtld_raw` keep the values for every song with English, so other thresholds (30, 100) can be applied later for robustness checks.

In [2]:
MIN_EN_TOKENS = 50     # minimum English tokens for TTR and MTLD
MTLD_THRESHOLD = 0.72  # standard MTLD factor threshold (McCarthy & Jarvis, 2010)

assert "is_romanized" in tokens, "tokens.pkl has no is_romanized column: re-run 05"

VERSIONS = {
    "all": tokens,
    "nofill": tokens[~tokens["is_filler"]],
    "romko": tokens.assign(lang=tokens["lang"].mask(tokens["is_romanized"], "ko")),
}
for name, t in VERSIONS.items():
    print(f"{name}: {len(t):,} tokens, {(t['lang'] == 'en').sum():,} English")

all: 1,666,993 tokens, 393,494 English
nofill: 1,637,401 tokens, 363,902 English
romko: 1,666,993 tokens, 393,478 English


## 3. Feature Computation

### 3.1 English Frequency

In [3]:
features = songs[["song_key"]].set_index("song_key")

for name, t in VERSIONS.items():
    counts = t.groupby(["song_key", "lang"]).size().unstack(fill_value=0).reindex(columns=["ko", "en"], fill_value=0)
    counts = counts.reindex(features.index, fill_value=0)
    features[f"n_ko_{name}"] = counts["ko"]
    features[f"n_en_{name}"] = counts["en"]
    total = counts["ko"] + counts["en"]
    features[f"en_ratio_{name}"] = (counts["en"] / total.replace(0, np.nan))

features["has_english"] = features["n_en_all"] > 0
print(f"Songs with any English: {features['has_english'].mean():.2%}")
display(features[[c for c in features if c.startswith("en_ratio")]].describe().round(3))

Songs with any English: 58.39%


,en_ratio_all,en_ratio_nofill,en_ratio_romko
count,7984.000,7984.000,7984.000
mean,0.172,0.164,0.172
std,0.238,0.233,0.238
min,0.000,0.000,0.000
25%,0.000,0.000,0.000
50%,0.048,0.040,0.048
75%,0.290,0.272,0.290
max,1.000,1.000,1.000


### 3.2 Lexical Diversity (TTR, MTLD)

#### Procedure
- English tokens of each song, in lyric order.
- TTR = distinct words / tokens.
- MTLD (McCarthy & Jarvis, 2010): read tokens in order and count a "factor" each time the running TTR drops to 0.72, then restart; the remainder adds a partial factor. MTLD = tokens / factors, averaged over the forward and backward passes.
- Computed only for songs with at least `MIN_EN_TOKENS` English tokens.

In [4]:
def _mtld_pass(words, threshold):
    factors, types, count, ttr = 0.0, set(), 0, 1.0
    for w in words:
        count += 1
        types.add(w)
        ttr = len(types) / count
        if ttr <= threshold:
            factors += 1
            types, count, ttr = set(), 0, 1.0
    if count > 0:
        factors += (1 - ttr) / (1 - threshold)
    return len(words) / factors if factors > 0 else np.nan

def mtld(words, threshold=MTLD_THRESHOLD):
    return (_mtld_pass(words, threshold) + _mtld_pass(words[::-1], threshold)) / 2

def ttr(words):
    return len(set(words)) / len(words)

for name, t in VERSIONS.items():
    en_words = t[t["lang"] == "en"].groupby("song_key")["token"].apply(list)
    features[f"ttr_raw_{name}"] = en_words.apply(ttr)
    features[f"mtld_raw_{name}"] = en_words.apply(mtld)
    eligible = features[f"n_en_{name}"] >= MIN_EN_TOKENS
    features[f"ttr_{name}"] = features[f"ttr_raw_{name}"].where(eligible)
    features[f"mtld_{name}"] = features[f"mtld_raw_{name}"].where(eligible)

print("Songs with TTR/MTLD by threshold (all tokens):")
n_en = features["n_en_all"]
display(pd.DataFrame({
    "min English tokens": [30, 50, 100],
    "songs": [(n_en >= k).sum() for k in [30, 50, 100]],
    "share of songs (%)": [round((n_en >= k).mean() * 100, 1) for k in [30, 50, 100]],
}).set_index("min English tokens"))
print(f"Current threshold: {MIN_EN_TOKENS}")

Songs with TTR/MTLD by threshold (all tokens):


,songs,share of songs (%)
min English tokens,,
30,3132,39.2
50,2500,31.3
100,1436,18.0


Current threshold: 50


### 3.3 English Phrase Length

#### Procedure
- A phrase is a run of consecutive English tokens within one line; a Korean token or a line break ends it.
- `phrase_len_mean`: mean run length; `phrase_len_max`: longest run.
- In the `_nofill` version, fillers are removed first, so `oh baby` counts as a 1-token phrase.

In [5]:
def phrase_lengths(t):
    t = t.copy()
    # New run starts when language changes or a new line starts
    new_run = (t["lang"] != t["lang"].shift()) | (t["line_no"] != t["line_no"].shift()) | (t["song_key"] != t["song_key"].shift())
    t["run_id"] = new_run.cumsum()
    runs = t[t["lang"] == "en"].groupby(["song_key", "run_id"]).size()
    return runs.groupby("song_key").agg(["mean", "max", "size"])

for name, t in VERSIONS.items():
    pl = phrase_lengths(t)
    features[f"phrase_len_mean_{name}"] = pl["mean"]
    features[f"phrase_len_max_{name}"] = pl["max"]
    features[f"n_phrases_{name}"] = pl["size"]

display(features[[c for c in features if c.startswith("phrase_len")]].describe().round(2))

,phrase_len_mean_all,phrase_len_max_all,phrase_len_mean_nofill,phrase_len_max_nofill,phrase_len_mean_romko,phrase_len_max_romko
count,4662.00,4662.00,4539.00,4539.00,4662.00,4662.00
mean,3.67,5.98,3.66,5.88,3.67,5.98
std,5.83,6.13,5.55,5.83,5.83,6.13
min,1.00,1.00,1.00,1.00,1.00,1.00
25%,2.67,4.00,2.71,4.00,2.67,4.00
50%,3.56,6.00,3.50,6.00,3.56,6.00
75%,4.39,7.00,4.38,7.00,4.39,7.00
max,370.00,370.00,345.00,345.00,370.00,370.00


### 3.4 Code-Switching

#### Procedure
- A switch is a change between Korean and English between adjacent tokens.
- **Within lines** (`cs_per100`): adjacent tokens in the same line, including inside mixed words such as `love해` (e.g. `널 원해 baby` = 1 switch).
- **At line breaks** (`cs_inter_per100`): last token of a line vs. first token of the next line (e.g. a Korean line followed by an English line = 1 switch).
- `cs_total_per100`: both together. All three are per 100 Korean/English tokens.
- `mixed_line_ratio`: share of lines that contain both languages.
- Lines with no Korean or English tokens are skipped, so the next lyric line counts as adjacent.

In [6]:
for name, t in VERSIONS.items():
    same_song = t["song_key"] == t["song_key"].shift()
    same_line = same_song & (t["line_no"] == t["line_no"].shift())
    lang_change = same_song & (t["lang"] != t["lang"].shift())
    n_switch = (lang_change & same_line).groupby(t["song_key"]).sum()
    n_inter = (lang_change & ~same_line).groupby(t["song_key"]).sum()
    n_tok = t.groupby("song_key").size()

    line_langs = t.groupby(["song_key", "line_no"])["lang"].nunique()
    mixed = (line_langs == 2).groupby("song_key").mean()

    features[f"n_switch_{name}"] = n_switch.reindex(features.index, fill_value=0)
    features[f"n_switch_inter_{name}"] = n_inter.reindex(features.index, fill_value=0)
    features[f"cs_per100_{name}"] = (n_switch / n_tok * 100).reindex(features.index)
    features[f"cs_inter_per100_{name}"] = (n_inter / n_tok * 100).reindex(features.index)
    features[f"cs_total_per100_{name}"] = ((n_switch + n_inter) / n_tok * 100).reindex(features.index)
    features[f"mixed_line_ratio_{name}"] = mixed.reindex(features.index)

display(features[[c for c in features if c.startswith(("cs_", "mixed_line"))]].describe().round(3))

print("Songs with at least one switch (all tokens):")
for col in ["cs_per100_all", "cs_inter_per100_all", "cs_total_per100_all"]:
    print(f"  {col}: {(features[col] > 0).mean():.2%}")

,cs_per100_all,cs_inter_per100_all,cs_total_per100_all,mixed_line_ratio_all,cs_per100_nofill,cs_inter_per100_nofill,cs_total_per100_nofill,mixed_line_ratio_nofill,cs_per100_romko,cs_inter_per100_romko,cs_total_per100_romko,mixed_line_ratio_romko
count,7984.000,7984.000,7984.000,7984.000,7984.000,7984.000,7984.000,7984.000,7984.000,7984.000,7984.000,7984.000
mean,1.522,3.200,4.723,0.055,1.354,3.128,4.482,0.048,1.523,3.200,4.723,0.055
std,2.670,3.955,5.930,0.096,2.531,3.986,5.813,0.089,2.672,3.955,5.932,0.096
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
50%,0.000,1.220,1.782,0.000,0.000,0.930,1.392,0.000,0.000,1.220,1.782,0.000
75%,2.163,5.958,8.571,0.078,1.778,5.830,8.130,0.064,2.163,5.958,8.571,0.078
max,34.722,22.816,44.722,1.000,34.819,23.529,44.847,1.000,34.722,22.816,44.722,1.000


Songs with at least one switch (all tokens):
  cs_per100_all: 43.84%
  cs_inter_per100_all: 55.01%
  cs_total_per100_all: 55.65%


### 3.5 Components of Lexical Diversity

#### Objective
Supplementary measures used in `07` (5.5) to interpret changes in lexical diversity. They describe *how* the English in a song is used: how much it repeats and which kinds of words it contains. Computed from all tokens for songs with at least one English token.

#### Repetition and vocabulary
| Measure | Definition |
|---|---|
| `en_types` | Distinct English words |
| `en_top_word_share` | Share of English tokens taken by the song's most frequent English word |
| `en_repeated_line_share` | Share of English tokens in lines whose full text appears more than once in the song (e.g. a repeated chorus) |
| `en_filler_share` | Share of English tokens that are fillers |

#### Part of speech
- Each English run (consecutive English tokens within a line, as in 3.3) is tagged with spaCy (`en_core_web_sm`). Tagging whole runs instead of single words lets the tagger use context.
- Each distinct run text is tagged once and weighted by how often it occurs in the song.
- Shares of English words (spaCy tokens, punctuation excluded) by group:

| Measure | spaCy tags | Examples |
|---|---|---|
| `pos_intj` | INTJ | oh, yeah, hey |
| `pos_noun` | NOUN, PROPN | love, girl, night |
| `pos_verb` | VERB | want, dance, know |
| `pos_adj_adv` | ADJ, ADV | crazy, so, never |
| `pos_pron` | PRON | I, you, me |
| `pos_function` | AUX, DET, ADP, CCONJ, SCONJ, PART | the, in, and, to, don't |
| `pos_other` | all other tags | |

Lyrics are fragmented and repetitive, so tagging accuracy is lower than on ordinary text; the shares are used for broad patterns.

In [7]:
import spacy

t = VERSIONS["all"]
en = t[t["lang"] == "en"]

# Repetition and vocabulary
line_text = t.groupby(["song_key", "line_no"])["token"].agg(" ".join)
line_repeats = line_text.groupby([line_text.index.get_level_values("song_key"), line_text.values]).transform("size")
en_in_repeated = line_repeats.reindex(pd.MultiIndex.from_frame(en[["song_key", "line_no"]])).to_numpy() > 1

g = en.groupby("song_key")
features["en_types"] = g["token"].nunique()
features["en_top_word_share"] = g["token"].agg(lambda s: s.value_counts().iloc[0] / len(s))
features["en_repeated_line_share"] = pd.Series(en_in_repeated, index=en.index).groupby(en["song_key"]).mean()
features["en_filler_share"] = g["is_filler"].mean()

# Part of speech of English runs
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "lemmatizer"])
POS_GROUPS = {"INTJ": "intj", "NOUN": "noun", "PROPN": "noun", "VERB": "verb", "ADJ": "adj_adv", "ADV": "adj_adv",
              "PRON": "pron", "AUX": "function", "DET": "function", "ADP": "function", "CCONJ": "function",
              "SCONJ": "function", "PART": "function"}
POS_COLS = ["intj", "noun", "verb", "adj_adv", "pron", "function", "other"]

new_run = (t["lang"] != t["lang"].shift()) | (t["line_no"] != t["line_no"].shift()) | (t["song_key"] != t["song_key"].shift())
runs = (t.assign(run_id=new_run.cumsum())[t["lang"] == "en"]
        .groupby(["song_key", "run_id"])["token"].agg(" ".join).rename("run").reset_index())
run_counts = runs.groupby(["song_key", "run"]).size().rename("n").reset_index()
distinct_runs = run_counts["run"].drop_duplicates().tolist()
print(f"English runs: {len(runs):,} ({len(distinct_runs):,} distinct)")

pos_rows = []
for doc in nlp.pipe(distinct_runs, batch_size=2000):
    counts = dict.fromkeys(POS_COLS, 0)
    for tok in doc:
        if not tok.is_punct:
            counts[POS_GROUPS.get(tok.pos_, "other")] += 1
    pos_rows.append(counts)
pos_run = pd.DataFrame(pos_rows, index=distinct_runs)

pos_song = (pos_run.reindex(run_counts["run"]).mul(run_counts["n"].to_numpy(), axis=0)
            .groupby(run_counts["song_key"].to_numpy()).sum())
pos_share = pos_song.div(pos_song.sum(axis=1).replace(0, np.nan), axis=0)
for c in POS_COLS:
    features[f"pos_{c}"] = pos_share[c]

COMPONENTS = ["en_types", "en_top_word_share", "en_repeated_line_share", "en_filler_share"] + [f"pos_{c}" for c in POS_COLS]
display(features[COMPONENTS].describe().round(3))

English runs: 102,377 (38,825 distinct)


,en_types,en_top_word_share,en_repeated_line_share,en_filler_share,pos_intj,pos_noun,pos_verb,pos_adj_adv,pos_pron,pos_function,pos_other
count,4662.000,4662.000,4662.000,4662.000,4662.000,4662.000,4662.000,4662.000,4662.000,4662.000,4662.000
mean,25.268,0.272,0.575,0.102,0.106,0.232,0.153,0.106,0.196,0.193,0.013
std,28.498,0.228,0.340,0.204,0.202,0.213,0.133,0.132,0.146,0.144,0.062
min,1.000,0.027,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,6.250,0.125,0.333,0.000,0.000,0.088,0.056,0.000,0.092,0.071,0.000
50%,16.000,0.198,0.667,0.017,0.025,0.185,0.145,0.075,0.190,0.194,0.000
75%,33.000,0.333,0.853,0.098,0.111,0.310,0.215,0.149,0.275,0.294,0.000
max,292.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000


## 4. Feature Checks

### 4.1 Summary Statistics

In [8]:
FEATURES = ["en_ratio", "ttr", "mtld", "phrase_len_mean", "phrase_len_max",
            "cs_per100", "cs_inter_per100", "cs_total_per100", "mixed_line_ratio"]

summary = pd.DataFrame({
    f"{f}_{v}": features[f"{f}_{v}"].describe() for f in FEATURES for v in VERSIONS
}).T.round(3)
display(summary[["count", "mean", "std", "min", "50%", "max"]])

,count,mean,std,min,50%,max
en_ratio_all,7984.0,0.172,0.238,0.000,0.048,1.000
en_ratio_nofill,7984.0,0.164,0.233,0.000,0.040,1.000
en_ratio_romko,7984.0,0.172,0.238,0.000,0.048,1.000
ttr_all,2500.0,0.295,0.148,0.008,0.277,0.900
ttr_nofill,2360.0,0.302,0.151,0.015,0.283,0.898
ttr_romko,2500.0,0.295,0.148,0.008,0.277,0.900
mtld_all,2500.0,16.963,16.386,2.000,11.676,168.000
mtld_nofill,2360.0,17.959,17.057,2.000,12.195,193.463
mtld_romko,2500.0,16.962,16.386,2.000,11.676,168.000
phrase_len_mean_all,4662.0,3.668,5.828,1.000,3.556,370.000


### 4.2 Length Sensitivity of TTR and MTLD

#### Objective
Show why MTLD is used alongside TTR: TTR should correlate negatively with the number of English tokens, while MTLD should be much less affected.

In [9]:
corr = pd.DataFrame({
    "TTR": [features["ttr_all"].corr(features["n_en_all"], method="spearman"),
            features["ttr_nofill"].corr(features["n_en_nofill"], method="spearman")],
    "MTLD": [features["mtld_all"].corr(features["n_en_all"], method="spearman"),
             features["mtld_nofill"].corr(features["n_en_nofill"], method="spearman")],
}, index=["all", "nofill"]).round(3)

print("Spearman correlation with the number of English tokens:")
display(corr)

Spearman correlation with the number of English tokens:


,TTR,MTLD
all,-0.106,0.188
nofill,-0.083,0.234


### 4.3 MTLD Implementation Check

#### Objective
Compare the MTLD implementation in 3.2 with the `lexicalrichness` package on a sample of songs. Skipped if the package is not installed.

In [10]:
try:
    from lexicalrichness import LexicalRichness

    en_words = VERSIONS["all"].query("lang == 'en'").groupby("song_key")["token"].apply(list)
    sample = en_words[en_words.str.len() >= MIN_EN_TOKENS].sample(min(200, (en_words.str.len() >= MIN_EN_TOKENS).sum()), random_state=0)

    ours = sample.apply(mtld)
    ref = sample.apply(lambda w: LexicalRichness(" ".join(w), preprocessor=None, tokenizer=str.split).mtld(threshold=MTLD_THRESHOLD))
    diff = (ours - ref).abs()
    print(f"Songs compared: {len(sample)}")
    print(f"Max absolute difference: {diff.max():.4f}, mean: {diff.mean():.4f}")
    print(f"Spearman correlation: {ours.corr(ref, method='spearman'):.4f}")
except ImportError:
    print("lexicalrichness is not installed; skipped (pip install lexicalrichness)")

Songs compared: 200
Max absolute difference: 0.0000, mean: 0.0000
Spearman correlation: 1.0000


### 4.4 Preview by Generation and Period

#### Objective
Quick look at medians before the statistical analysis in the next notebook. No tests are run here.

In [11]:
df = songs.set_index("song_key").join(features)

preview_cols = [f"{f}_all" for f in FEATURES]

print("Median by generation (baseline, RQ1 songs):")
display(df[df["in_rq1"]].groupby("gen_baseline", observed=True)[preview_cols].median().round(3))

period = pd.cut(df["release_year"], bins=[1999, 2004, 2009, 2014, 2019, 2023],
                labels=["2000–04", "2005–09", "2010–14", "2015–19", "2020–23"])
print("Median by release period (RQ2 songs):")
display(df[df["in_rq2"]].groupby(period[df["in_rq2"]], observed=True)[preview_cols].median().round(3))

Median by generation (baseline, RQ1 songs):


,en_ratio_all,ttr_all,mtld_all,phrase_len_mean_all,phrase_len_max_all,cs_per100_all,cs_inter_per100_all,cs_total_per100_all,mixed_line_ratio_all
gen_baseline,,,,,,,,,
1st,0.000,0.293,14.400,3.591,6.0,0.000,0.000,0.000,0.000
2nd,0.070,0.256,9.504,3.579,6.0,0.000,2.210,3.211,0.000
3rd,0.181,0.272,10.639,3.417,6.0,0.973,4.666,6.597,0.036
4th,0.471,0.308,12.883,3.560,7.0,3.671,7.505,11.549,0.128


Median by release period (RQ2 songs):


,en_ratio_all,ttr_all,mtld_all,phrase_len_mean_all,phrase_len_max_all,cs_per100_all,cs_inter_per100_all,cs_total_per100_all,mixed_line_ratio_all
release_year,,,,,,,,,
2000–04,0.000,0.321,19.988,3.750,6.0,0.000,0.000,0.000,0.000
2005–09,0.001,0.279,12.803,3.667,6.0,0.000,0.000,0.000,0.000
2010–14,0.062,0.241,9.087,3.562,6.0,0.000,1.622,2.492,0.000
2015–19,0.138,0.266,10.204,3.392,6.0,0.775,4.000,5.639,0.026
2020–23,0.141,0.312,15.308,3.637,7.0,0.612,2.460,4.019,0.023


## 5. Export

### Output
- `data/interim/features.pkl`: one row per song with labels (from `04`) and all features

In [12]:
label_columns = [
    "song_name", "artist", "release_year", "first_chart_year", "n_chart_months", "peak_rank",
    "artist_type", "type_group", "debut_year", "confidence",
    "gen_baseline", "gen_alt_A", "gen_alt_B", "gen_alt_C", "high_conf", "in_rq1", "in_rq2",
]
out = df[label_columns].join(features).reset_index()
out.to_pickle(INTERIM_DIR / "features.pkl")

print(f"Saved {len(out):,} songs with {features.shape[1]} feature columns to {INTERIM_DIR / 'features.pkl'}")

Saved 7,984 songs with 60 feature columns to ../data/interim/features.pkl
